In [1]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)

Train shape: (428067, 5)
Test shape: (106980, 5)


In [2]:
# Jedinstveni korisnici i recepti
user_ids = train_ratings["user_id"].unique()
recipe_ids = train_ratings["recipe_id"].unique()

num_users = len(user_ids)
num_recipes = len(recipe_ids)

print("Number of users:", num_users)
print("Number of recipes:", num_recipes)

Number of users: 17034
Number of recipes: 40027


In [3]:
# Mapiranje originalnih ID-jeva na indekse za neuronsku mrežu

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

recipe_to_index = {
    recipe_id: index
    for index, recipe_id in enumerate(recipe_ids)
}

print("User mapping:", list(user_to_index.items())[:5])
print("Recipe mapping:", list(recipe_to_index.items())[:5])

User mapping: [(np.int64(450571), 0), (np.int64(785604), 1), (np.int64(573325), 2), (np.int64(245429), 3), (np.int64(56112), 4)]
Recipe mapping: [(np.int64(38067), 0), (np.int64(31925), 1), (np.int64(213987), 2), (np.int64(110408), 3), (np.int64(76930), 4)]


In [4]:
# Kopija trening podataka
ncf_train = train_ratings[["user_id", "recipe_id", "rating"]].copy()

# Originalne ID-jeve pretvaramo u indekse
ncf_train["user_index"] = ncf_train["user_id"].map(user_to_index)
ncf_train["recipe_index"] = ncf_train["recipe_id"].map(recipe_to_index)

print(ncf_train.head())
print("Training samples:", len(ncf_train))

   user_id  recipe_id  rating  user_index  recipe_index
0   450571      38067       5           0             0
1   785604      31925       5           1             1
2   573325     213987       5           2             2
3   245429     110408       5           3             3
4    56112      76930       4           4             4
Training samples: 428067


In [5]:
print("Missing user indexes:", ncf_train["user_index"].isna().sum())
print("Missing recipe indexes:", ncf_train["recipe_index"].isna().sum())

Missing user indexes: 0
Missing recipe indexes: 0


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, dataframe):
        self.users = torch.tensor(
            dataframe["user_index"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            dataframe["recipe_index"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            dataframe["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, index):
        return (
            self.users[index],
            self.recipes[index],
            self.ratings[index]
        )

In [9]:
train_dataset = RecipeRatingDataset(ncf_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

print("Number of training samples:", len(train_dataset))

Number of training samples: 428067


In [10]:
users, recipes_batch, ratings = next(iter(train_loader))

print("Users shape:", users.shape)
print("Recipes shape:", recipes_batch.shape)
print("Ratings shape:", ratings.shape)

print("\nFirst 5 users:", users[:5])
print("First 5 recipes:", recipes_batch[:5])
print("First 5 ratings:", ratings[:5])

Users shape: torch.Size([256])
Recipes shape: torch.Size([256])
Ratings shape: torch.Size([256])

First 5 users: tensor([  33, 2068,  762, 6778, 5821])
First 5 recipes: tensor([ 2013,  4968, 19741, 32859, 10174])
First 5 ratings: tensor([5., 5., 5., 5., 5.])


In [11]:
import torch.nn as nn


class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32
    ):
        super().__init__()

        # User embedding
        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        # Recipe embedding
        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        # Neural network
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        # Spojimo user i recipe embedding
        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        # Neural network
        output = self.mlp(x)

        return output.squeeze(1)

In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32
).to(device)

print(model)

Using device: cpu
NCF(
  (user_embedding): Embedding(17034, 32)
  (recipe_embedding): Embedding(40027, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [13]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model parameters:", sum(
    p.numel()
    for p in model.parameters()
))

Model parameters: 1832737


In [14]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU nije dostupan za PyTorch.")

CUDA available: False
GPU nije dostupan za PyTorch.


In [15]:
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, ratings):
        self.users = torch.tensor(
            ratings["user_idx"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            ratings["recipe_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            ratings["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.recipes[idx],
            self.ratings[idx]
        )

In [16]:
print(train_ratings.columns.tolist())

['user_id', 'recipe_id', 'date', 'rating', 'review']


In [17]:
# Mapiranje stvarnih ID-jeva na indekse koje PyTorch Embedding koristi

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(train_ratings["user_id"].unique())
}

recipe_to_idx = {
    recipe_id: idx
    for idx, recipe_id in enumerate(train_ratings["recipe_id"].unique())
}

# Dodaj indekse u training skup
train_ratings["user_idx"] = train_ratings["user_id"].map(user_to_idx)
train_ratings["recipe_idx"] = train_ratings["recipe_id"].map(recipe_to_idx)

print("Number of users:", len(user_to_idx))
print("Number of recipes:", len(recipe_to_idx))

print(train_ratings[
          ["user_id", "user_idx", "recipe_id", "recipe_idx", "rating"]
      ].head())

Number of users: 17034
Number of recipes: 40027
   user_id  user_idx  recipe_id  recipe_idx  rating
0   450571         0      38067           0       5
1   785604         1      31925           1       5
2   573325         2     213987           2       5
3   245429         3     110408           3       5
4    56112         4      76930           4       4


In [18]:

print("Da li je GPU dostupan:", torch.cuda.is_available())
     

Da li je GPU dostupan: False


In [19]:
from torch.utils.data import Dataset, DataLoader


class RecipeRatingDataset(Dataset):

    def __init__(self, ratings):
        self.users = torch.tensor(
            ratings["user_idx"].values,
            dtype=torch.long
        )

        self.recipes = torch.tensor(
            ratings["recipe_idx"].values,
            dtype=torch.long
        )

        self.ratings = torch.tensor(
            ratings["rating"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return (
            self.users[idx],
            self.recipes[idx],
            self.ratings[idx]
        )


train_dataset = RecipeRatingDataset(train_ratings)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

print("Dataset size:", len(train_dataset))
print("Number of batches:", len(train_loader))

Dataset size: 428067
Number of batches: 419


In [20]:
users, recipes_batch, ratings_batch = next(iter(train_loader))

print("Users shape:", users.shape)
print("Recipes shape:", recipes_batch.shape)
print("Ratings shape:", ratings_batch.shape)

print("\nFirst 5 users:", users[:5])
print("First 5 recipes:", recipes_batch[:5])
print("First 5 ratings:", ratings_batch[:5])

Users shape: torch.Size([1024])
Recipes shape: torch.Size([1024])
Ratings shape: torch.Size([1024])

First 5 users: tensor([ 2753,    18,   589, 10355,   547])
First 5 recipes: tensor([13339,  3841,  2554,   481,  2730])
First 5 ratings: tensor([5., 5., 5., 5., 5.])


In [21]:
model.eval()

with torch.no_grad():

    predictions = model(
        users.to(device),
        recipes_batch.to(device)
    )

print("Predictions shape:", predictions.shape)
print("First 10 predictions:", predictions[:10])

Predictions shape: torch.Size([1024])
First 10 predictions: tensor([0.2309, 0.2079, 0.2390, 0.1659, 0.2354, 0.2172, 0.2259, 0.2313, 0.2285,
        0.2328])


In [24]:
import time

num_epochs = 5

model.train()

for epoch in range(num_epochs):

    start_time = time.time()

    total_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        # Predikcija
        predictions = model(
            users,
            recipes_batch
        )

        # Greška
        loss = criterion(
            predictions,
            ratings_batch
        )

        # Reset gradienta
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parametara
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Loss: {avg_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

Epoch 1/5 - Loss: 0.3060 - Time: 10.8s
Epoch 2/5 - Loss: 0.2948 - Time: 9.2s
Epoch 3/5 - Loss: 0.2849 - Time: 10.1s
Epoch 4/5 - Loss: 0.2757 - Time: 10.0s
Epoch 5/5 - Loss: 0.2665 - Time: 9.3s


In [23]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\Scripts\python.exe
PyTorch: 2.13.0+cpu
CUDA available: False
CUDA version: None


In [25]:
test_ratings["user_idx"] = test_ratings["user_id"].map(user_to_idx)
test_ratings["recipe_idx"] = test_ratings["recipe_id"].map(recipe_to_idx)

print("Test shape:", test_ratings.shape)
print(
    "Users missing:",
    test_ratings["user_idx"].isna().sum()
)
print(
    "Recipes missing:",
    test_ratings["recipe_idx"].isna().sum()
)

Test shape: (106980, 7)
Users missing: 0
Recipes missing: 0


In [26]:
test_dataset = RecipeRatingDataset(test_ratings)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False
)

print("Test samples:", len(test_dataset))
print("Test batches:", len(test_loader))

Test samples: 106980
Test batches: 105


In [27]:
model.eval()

total_test_loss = 0.0

with torch.no_grad():

    for users, recipes_batch, ratings_batch in test_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        total_test_loss += loss.item()

avg_test_loss = total_test_loss / len(test_loader)

print(f"Test Loss: {avg_test_loss:.4f}")

Test Loss: 0.3419


In [28]:
# Recepti koje je svaki korisnik već ocenio u trening skupu
user_seen_recipes = (
    train_ratings
    .groupby("user_idx")["recipe_idx"]
    .apply(set)
    .to_dict()
)

print("Users with training history:", len(user_seen_recipes))

Users with training history: 17034


In [29]:
def recommend_ncf(user_idx, model, num_recipes, user_seen_recipes, top_k=10):

    model.eval()

    seen = user_seen_recipes.get(user_idx, set())

    # Kandidati su recepti koje korisnik još nije ocenio
    candidate_recipes = [
        recipe_idx
        for recipe_idx in range(num_recipes)
        if recipe_idx not in seen
    ]

    users = torch.tensor(
        [user_idx] * len(candidate_recipes),
        dtype=torch.long
    ).to(device)

    recipes = torch.tensor(
        candidate_recipes,
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        predictions = model(users, recipes)

    # Top-K
    top_indices = torch.topk(
        predictions,
        k=top_k
    ).indices

    recommended_recipes = [
        candidate_recipes[i]
        for i in top_indices.cpu().numpy()
    ]

    return recommended_recipes

In [30]:
# Uzimamo prvog korisnika koji postoji u test skupu
test_user = test_ratings["user_idx"].iloc[0]

recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Recommended recipe indexes:", recommendations)

User: 3725
Recommended recipe indexes: [12323, 16285, 35563, 21863, 13711, 25890, 30894, 6361, 18098, 20664]


In [31]:
from sklearn.model_selection import train_test_split

# Odvajamo 10% trening podataka za validation
train_data, val_data = train_test_split(
    train_ratings,
    test_size=0.10,
    random_state=42
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

Train: (385260, 7)
Validation: (42807, 7)


In [32]:
train_dataset = RecipeRatingDataset(train_data)
val_dataset = RecipeRatingDataset(val_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Training samples: 385260
Validation samples: 42807


In [33]:
import torch.nn as nn


class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32,
            dropout=0.2
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        output = self.mlp(x)

        return output.squeeze(1)

In [34]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32,
    dropout=0.2
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Device:", device)
print("Parameters:", sum(
    p.numel() for p in model.parameters()
))

Device: cpu
Parameters: 1832737


In [36]:
import time
import copy

num_epochs = 35
patience = 7

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================

    model.train()

    total_train_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for users, recipes_batch, ratings_batch in val_loader:

            users = users.to(device)
            recipes_batch = recipes_batch.to(device)
            ratings_batch = ratings_batch.to(device)

            predictions = model(
                users,
                recipes_batch
            )

            loss = criterion(
                predictions,
                ratings_batch
            )

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # =========================
    # BEST MODEL
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        print("  -> New best model!")

    else:

        epochs_without_improvement += 1

        print(
            f"  -> No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

    # =========================
    # EARLY STOPPING
    # =========================

    if epochs_without_improvement >= patience:

        print("Early stopping triggered.")

        break

Epoch 01/35 - Train Loss: 0.3218 - Val Loss: 0.3206 - Time: 11.7s
  -> New best model!
Epoch 02/35 - Train Loss: 0.3028 - Val Loss: 0.3176 - Time: 10.3s
  -> New best model!
Epoch 03/35 - Train Loss: 0.2860 - Val Loss: 0.3167 - Time: 10.5s
  -> New best model!
Epoch 04/35 - Train Loss: 0.2727 - Val Loss: 0.3224 - Time: 10.0s
  -> No improvement (1/7)
Epoch 05/35 - Train Loss: 0.2641 - Val Loss: 0.3282 - Time: 9.7s
  -> No improvement (2/7)
Epoch 06/35 - Train Loss: 0.2565 - Val Loss: 0.3324 - Time: 9.7s
  -> No improvement (3/7)
Epoch 07/35 - Train Loss: 0.2499 - Val Loss: 0.3405 - Time: 9.9s
  -> No improvement (4/7)
Epoch 08/35 - Train Loss: 0.2421 - Val Loss: 0.3506 - Time: 9.8s
  -> No improvement (5/7)
Epoch 09/35 - Train Loss: 0.2342 - Val Loss: 0.3545 - Time: 9.6s
  -> No improvement (6/7)
Epoch 10/35 - Train Loss: 0.2247 - Val Loss: 0.3616 - Time: 9.9s
  -> No improvement (7/7)
Early stopping triggered.


In [37]:
class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=16,
            dropout=0.3
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        output = self.mlp(x)

        return output.squeeze(1)

In [38]:
model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=16,
    dropout=0.3
).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

print("Device:", device)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Device: cpu
Parameters: 914705


In [39]:
import time
import copy

num_epochs = 35
patience = 7

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    start_time = time.time()

    # =========================
    # TRAIN
    # =========================

    model.train()

    total_train_loss = 0.0

    for users, recipes_batch, ratings_batch in train_loader:

        users = users.to(device)
        recipes_batch = recipes_batch.to(device)
        ratings_batch = ratings_batch.to(device)

        optimizer.zero_grad()

        predictions = model(
            users,
            recipes_batch
        )

        loss = criterion(
            predictions,
            ratings_batch
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    total_val_loss = 0.0

    with torch.no_grad():

        for users, recipes_batch, ratings_batch in val_loader:

            users = users.to(device)
            recipes_batch = recipes_batch.to(device)
            ratings_batch = ratings_batch.to(device)

            predictions = model(
                users,
                recipes_batch
            )

            loss = criterion(
                predictions,
                ratings_batch
            )

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Loss: {avg_val_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # =========================
    # BEST MODEL
    # =========================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        print("  -> New best model!")

    else:

        epochs_without_improvement += 1

        print(
            f"  -> No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

    # =========================
    # EARLY STOPPING
    # =========================

    if epochs_without_improvement >= patience:

        print("Early stopping triggered.")

        break

Epoch 01/35 - Train Loss: 8.2180 - Val Loss: 0.8270 - Time: 10.9s
  -> New best model!
Epoch 02/35 - Train Loss: 1.9869 - Val Loss: 0.6432 - Time: 10.9s
  -> New best model!
Epoch 03/35 - Train Loss: 1.6067 - Val Loss: 0.5280 - Time: 11.1s
  -> New best model!
Epoch 04/35 - Train Loss: 1.2901 - Val Loss: 0.4373 - Time: 9.4s
  -> New best model!
Epoch 05/35 - Train Loss: 1.0747 - Val Loss: 0.4043 - Time: 7.7s
  -> New best model!
Epoch 06/35 - Train Loss: 0.9102 - Val Loss: 0.3884 - Time: 9.2s
  -> New best model!
Epoch 07/35 - Train Loss: 0.7839 - Val Loss: 0.3732 - Time: 8.1s
  -> New best model!
Epoch 08/35 - Train Loss: 0.6928 - Val Loss: 0.3821 - Time: 8.1s
  -> No improvement (1/7)
Epoch 09/35 - Train Loss: 0.6188 - Val Loss: 0.3703 - Time: 8.3s
  -> New best model!
Epoch 10/35 - Train Loss: 0.5482 - Val Loss: 0.3706 - Time: 7.9s
  -> No improvement (1/7)
Epoch 11/35 - Train Loss: 0.4837 - Val Loss: 0.3697 - Time: 8.9s
  -> New best model!
Epoch 12/35 - Train Loss: 0.4315 - Val Lo

In [40]:
model.load_state_dict(best_model_state)
model.eval()

print(f"Best validation loss: {best_val_loss:.4f}")

Best validation loss: 0.3162


In [41]:
# Korisnici koji imaju istoriju u trening skupu
train_user_counts = train_data.groupby("user_idx").size()

# Korisnici koji imaju bar jedan validation rating
val_users = val_data["user_idx"].unique()

# Zadržavamo korisnike sa dovoljno istorije
eval_users = [
    user for user in val_users
    if train_user_counts.get(user, 0) >= 5
]

print("Validation users:", len(val_users))
print("Evaluation users:", len(eval_users))

Validation users: 10711
Evaluation users: 8821


In [42]:
def recommend_ncf(
        user_idx,
        model,
        num_recipes,
        user_seen_recipes,
        top_k=10,
        recipe_batch_size=2048
):
    model.eval()

    seen = user_seen_recipes.get(user_idx, set())

    all_scores = []

    with torch.no_grad():

        for start in range(0, num_recipes, recipe_batch_size):

            end = min(
                start + recipe_batch_size,
                num_recipes
            )

            candidate_recipes = [
                r for r in range(start, end)
                if r not in seen
            ]

            if not candidate_recipes:
                continue

            users = torch.full(
                (len(candidate_recipes),),
                user_idx,
                dtype=torch.long,
                device=device
            )

            recipes = torch.tensor(
                candidate_recipes,
                dtype=torch.long,
                device=device
            )

            scores = model(users, recipes)

            all_scores.extend(
                zip(
                    candidate_recipes,
                    scores.cpu().numpy()
                )
            )

    # Sortiranje po predviđenoj oceni
    all_scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        recipe_idx
        for recipe_idx, score in all_scores[:top_k]
    ]

In [43]:
test_user = eval_users[0]

recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Top-10:", recommendations)

User: 11035
Top-10: [2536, 29685, 36952, 12609, 12748, 3736, 23000, 30990, 15129, 3796]


In [44]:
user_val = val_data[
    (val_data["user_idx"] == test_user) &
    (val_data["rating"] >= 4)
    ]

relevant_recipes = set(
    user_val["recipe_idx"]
)

print("Relevant recipes:", relevant_recipes)
print("Recommended:", set(recommendations))

Relevant recipes: {8096, 31172, 35167, 31147, 19836, 34303}
Recommended: {12609, 23000, 2536, 12748, 3736, 30990, 3796, 29685, 36952, 15129}


In [45]:
hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if len(relevant_recipes) > 0
    else 0
)

print(f"Hits: {hits}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Hits: 0
Precision@10: 0.0000
Recall@10: 0.0000


In [46]:
print("User:", test_user)

print("\nValidation ratings:")
print(
    val_data[
        val_data["user_idx"] == test_user
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
    .head(20)
)

print("\nNCF recommendations:")
print(recommendations)

User: 11035

Validation ratings:
       recipe_idx  rating
1           19836       5
1807         8096       5
4427        31147       5
12225       35167       5
30494       34303       4
32210       31172       4
8714         1877       3

NCF recommendations:
[2536, 29685, 36952, 12609, 12748, 3736, 23000, 30990, 15129, 3796]


In [47]:
# Prvih 20 preporuka sa njihovim score-ovima

model.eval()

seen = user_seen_recipes.get(test_user, set())

scores = []

with torch.no_grad():
    for recipe_idx in range(num_recipes):

        if recipe_idx in seen:
            continue

        user_tensor = torch.tensor(
            [test_user],
            dtype=torch.long,
            device=device
        )

        recipe_tensor = torch.tensor(
            [recipe_idx],
            dtype=torch.long,
            device=device
        )

        score = model(
            user_tensor,
            recipe_tensor
        ).item()

        scores.append(
            (recipe_idx, score)
        )

scores.sort(
    key=lambda x: x[1],
    reverse=True
)

print("Top 20 predictions:")
for recipe_idx, score in scores[:20]:
    print(
        f"Recipe {recipe_idx}: "
        f"{score:.4f}"
    )

Top 20 predictions:
Recipe 2536: 4.9787
Recipe 29685: 4.9676
Recipe 36952: 4.9473
Recipe 12609: 4.9402
Recipe 12748: 4.9295
Recipe 3736: 4.9117
Recipe 23000: 4.9098
Recipe 30990: 4.9094
Recipe 15129: 4.9082
Recipe 3796: 4.9052
Recipe 17608: 4.9019
Recipe 7345: 4.8991
Recipe 7858: 4.8986
Recipe 5891: 4.8974
Recipe 32217: 4.8958
Recipe 5830: 4.8951
Recipe 257: 4.8915
Recipe 25907: 4.8904
Recipe 14772: 4.8882
Recipe 17368: 4.8868


In [48]:
# Relevantni recepti korisnika iz validation skupa
relevant_recipes = set(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ]["recipe_idx"]
)

print("Relevant recipes:", relevant_recipes)

# Pozicija svakog relevantnog recepta u NCF rankingu
recipe_ranks = {
    recipe_idx: rank
    for rank, (recipe_idx, score) in enumerate(scores, start=1)
}

print("\nRank relevantnih recepata:")

for recipe in relevant_recipes:
    rank = recipe_ranks.get(recipe)

    if rank is not None:
        print(
            f"Recipe {recipe}: "
            f"Rank = {rank}"
        )

Relevant recipes: {8096, 31172, 35167, 31147, 19836, 34303}

Rank relevantnih recepata:


In [49]:
for recipe in relevant_recipes:
    print(
        f"Recipe {recipe}: "
        f"seen = {recipe in user_seen_recipes.get(test_user, set())}"
    )

Recipe 8096: seen = True
Recipe 31172: seen = True
Recipe 35167: seen = True
Recipe 31147: seen = True
Recipe 19836: seen = True
Recipe 34303: seen = True


In [50]:
user_seen_recipes = (
    train_data
    .groupby("user_idx")["recipe_idx"]
    .apply(set)
    .to_dict()
)

print("user_seen_recipes recreated from TRAIN only.")

user_seen_recipes recreated from TRAIN only.


In [51]:
for recipe in relevant_recipes:
    print(
        f"Recipe {recipe}: "
        f"seen = {recipe in user_seen_recipes.get(test_user, set())}"
    )

Recipe 8096: seen = False
Recipe 31172: seen = False
Recipe 35167: seen = False
Recipe 31147: seen = False
Recipe 19836: seen = False
Recipe 34303: seen = False


In [52]:
recommendations = recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("Recommendations:")
print(recommendations)

Recommendations:
[2536, 29685, 36952, 12609, 12748, 3736, 23000, 30990, 15129, 3796]


In [53]:
hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if len(relevant_recipes) > 0
    else 0
)

print(f"Hits: {hits}")
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Hits: 0
Precision@10: 0.0000
Recall@10: 0.0000


In [54]:
import numpy as np
import time

model.eval()

precisions = []
recalls = []
hit_counts = []

start_time = time.time()

for i, user_idx in enumerate(eval_users):

    # Relevantni recepti iz validation skupa
    relevant = set(
        val_data[
            (val_data["user_idx"] == user_idx) &
            (val_data["rating"] >= 4)
            ]["recipe_idx"]
    )

    # Ako korisnik nema relevantan validation recept,
    # preskačemo ga
    if len(relevant) == 0:
        continue

    # Top-10 preporuke
    recommended = recommend_ncf(
        user_idx=user_idx,
        model=model,
        num_recipes=num_recipes,
        user_seen_recipes=user_seen_recipes,
        top_k=10
    )

    recommended = set(recommended)

    hits = len(recommended & relevant)

    precision = hits / 10
    recall = hits / len(relevant)

    precisions.append(precision)
    recalls.append(recall)
    hit_counts.append(hits)

    if (i + 1) % 100 == 0:
        print(
            f"Processed {i + 1}/{len(eval_users)} users"
        )

elapsed = time.time() - start_time

print("\n==============================")
print("NCF EVALUATION")
print("==============================")

print(f"Users evaluated: {len(precisions)}")
print(f"Precision@10: {np.mean(precisions):.4f}")
print(f"Recall@10:    {np.mean(recalls):.4f}")
print(f"Average Hits: {np.mean(hit_counts):.4f}")
print(f"Time: {elapsed:.1f}s")

Processed 100/8821 users
Processed 200/8821 users
Processed 300/8821 users
Processed 400/8821 users
Processed 500/8821 users
Processed 700/8821 users
Processed 800/8821 users
Processed 900/8821 users
Processed 1000/8821 users
Processed 1100/8821 users
Processed 1200/8821 users
Processed 1300/8821 users
Processed 1400/8821 users
Processed 1500/8821 users
Processed 1600/8821 users
Processed 1700/8821 users
Processed 1800/8821 users
Processed 1900/8821 users
Processed 2000/8821 users
Processed 2100/8821 users
Processed 2200/8821 users
Processed 2300/8821 users
Processed 2400/8821 users
Processed 2500/8821 users
Processed 2600/8821 users
Processed 2700/8821 users
Processed 2800/8821 users
Processed 2900/8821 users
Processed 3000/8821 users
Processed 3100/8821 users
Processed 3200/8821 users
Processed 3300/8821 users
Processed 3400/8821 users
Processed 3500/8821 users
Processed 3600/8821 users
Processed 3700/8821 users
Processed 3800/8821 users
Processed 3900/8821 users
Processed 4000/8821 

In [55]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class BPRDataset(Dataset):

    def __init__(
            self,
            ratings_df,
            num_recipes
    ):
        self.num_recipes = num_recipes

        # Pozitivni recepti: rating >= 4
        positive_df = ratings_df[
            ratings_df["rating"] >= 4
            ].copy()

        self.positive_pairs = list(
            zip(
                positive_df["user_idx"].astype(int),
                positive_df["recipe_idx"].astype(int)
            )
        )

        # Recepti koje je korisnik već ocenio
        self.user_seen = (
            ratings_df
            .groupby("user_idx")["recipe_idx"]
            .apply(set)
            .to_dict()
        )

    def __len__(self):
        return len(self.positive_pairs)

    def __getitem__(self, idx):

        user, positive_recipe = self.positive_pairs[idx]

        seen = self.user_seen.get(user, set())

        # Nasumičan negativan recept
        negative_recipe = np.random.randint(
            0,
            self.num_recipes
        )

        while negative_recipe in seen:
            negative_recipe = np.random.randint(
                0,
                self.num_recipes
            )

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(
                positive_recipe,
                dtype=torch.long
            ),
            torch.tensor(
                negative_recipe,
                dtype=torch.long
            )
        )

In [56]:
bpr_dataset = BPRDataset(
    train_data,
    num_recipes
)

bpr_loader = DataLoader(
    bpr_dataset,
    batch_size=1024,
    shuffle=True
)

print("BPR samples:", len(bpr_dataset))

BPR samples: 367581


In [57]:
class NCF(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            embedding_dim=32,
            dropout=0.2
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 16),
            nn.ReLU(),

            nn.Linear(16, 1)
        )

    def forward(self, user, recipe):

        user_vector = self.user_embedding(user)
        recipe_vector = self.recipe_embedding(recipe)

        x = torch.cat(
            [user_vector, recipe_vector],
            dim=1
        )

        return self.mlp(x).squeeze(1)

In [58]:
def bpr_loss(
        positive_scores,
        negative_scores
):
    return -torch.mean(
        torch.log(
            torch.sigmoid(
                positive_scores - negative_scores
            ) + 1e-8
        )
    )

In [59]:
model = NCF(
    num_users=num_users,
    num_recipes=num_recipes,
    embedding_dim=32,
    dropout=0.2
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

print("Device:", device)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Device: cpu
Parameters: 1832737


In [60]:
num_epochs = 35

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for users, positive_recipes, negative_recipes in bpr_loader:

        users = users.to(device)
        positive_recipes = positive_recipes.to(device)
        negative_recipes = negative_recipes.to(device)

        optimizer.zero_grad()

        positive_scores = model(
            users,
            positive_recipes
        )

        negative_scores = model(
            users,
            negative_recipes
        )

        loss = bpr_loss(
            positive_scores,
            negative_scores
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(bpr_loader)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- BPR Loss: {avg_loss:.4f}"
    )

Epoch 01/35 - BPR Loss: 0.6924
Epoch 02/35 - BPR Loss: 0.6651
Epoch 03/35 - BPR Loss: 0.6096
Epoch 04/35 - BPR Loss: 0.5738
Epoch 05/35 - BPR Loss: 0.5522
Epoch 06/35 - BPR Loss: 0.5334
Epoch 07/35 - BPR Loss: 0.5190
Epoch 08/35 - BPR Loss: 0.5064
Epoch 09/35 - BPR Loss: 0.4912
Epoch 10/35 - BPR Loss: 0.4739
Epoch 11/35 - BPR Loss: 0.4554
Epoch 12/35 - BPR Loss: 0.4403
Epoch 13/35 - BPR Loss: 0.4246
Epoch 14/35 - BPR Loss: 0.4093
Epoch 15/35 - BPR Loss: 0.3975
Epoch 16/35 - BPR Loss: 0.3862
Epoch 17/35 - BPR Loss: 0.3744
Epoch 18/35 - BPR Loss: 0.3647
Epoch 19/35 - BPR Loss: 0.3561
Epoch 20/35 - BPR Loss: 0.3473
Epoch 21/35 - BPR Loss: 0.3401
Epoch 22/35 - BPR Loss: 0.3327
Epoch 23/35 - BPR Loss: 0.3271
Epoch 24/35 - BPR Loss: 0.3215
Epoch 25/35 - BPR Loss: 0.3158
Epoch 26/35 - BPR Loss: 0.3095
Epoch 27/35 - BPR Loss: 0.3057
Epoch 28/35 - BPR Loss: 0.3005
Epoch 29/35 - BPR Loss: 0.2966
Epoch 30/35 - BPR Loss: 0.2920
Epoch 31/35 - BPR Loss: 0.2892
Epoch 32/35 - BPR Loss: 0.2852
Epoch 33

In [61]:
model.eval()

with torch.no_grad():
    recipe_embeddings = model.recipe_embedding.weight

print(recipe_embeddings.shape)

torch.Size([40027, 32])


In [62]:
def fast_recommend_ncf(
        user_idx,
        model,
        num_recipes,
        user_seen_recipes,
        top_k=10
):
    model.eval()

    seen = user_seen_recipes.get(
        user_idx,
        set()
    )

    with torch.no_grad():

        user_tensor = torch.tensor(
            [user_idx],
            dtype=torch.long,
            device=device
        )

        user_embedding = model.user_embedding(
            user_tensor
        )

        user_embedding = user_embedding.expand(
            num_recipes,
            -1
        )

        recipe_indices = torch.arange(
            num_recipes,
            device=device
        )

        recipe_embedding = model.recipe_embedding(
            recipe_indices
        )

        x = torch.cat(
            [
                user_embedding,
                recipe_embedding
            ],
            dim=1
        )

        scores = model.mlp(x).squeeze(1)

        # Izbacujemo recepte koje je korisnik već
        # imao u TRAIN skupu
        if seen:
            seen_tensor = torch.tensor(
                list(seen),
                dtype=torch.long,
                device=device
            )

            scores[seen_tensor] = -float("inf")

        top_scores, top_indices = torch.topk(
            scores,
            k=top_k
        )

    return top_indices.cpu().numpy()

In [63]:
test_user = eval_users[0]

recommendations = fast_recommend_ncf(
    user_idx=test_user,
    model=model,
    num_recipes=num_recipes,
    user_seen_recipes=user_seen_recipes,
    top_k=10
)

print("User:", test_user)
print("Top-10:", recommendations)

User: 11035
Top-10: [ 457  237  593  410  143 2332  959  600 2181  216]


In [64]:
relevant_recipes = set(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ]["recipe_idx"]
)

hits = len(
    set(recommendations) & relevant_recipes
)

precision_at_10 = hits / 10

recall_at_10 = (
    hits / len(relevant_recipes)
    if relevant_recipes
    else 0
)

print("Relevant:", relevant_recipes)
print("Recommended:", recommendations)
print("Hits:", hits)
print(f"Precision@10: {precision_at_10:.4f}")
print(f"Recall@10: {recall_at_10:.4f}")

Relevant: {8096, 31172, 35167, 31147, 19836, 34303}
Recommended: [ 457  237  593  410  143 2332  959  600 2181  216]
Hits: 0
Precision@10: 0.0000
Recall@10: 0.0000


In [65]:
print("USER:", test_user)

print("\nTRAIN positives:")
print(
    train_data[
        (train_data["user_idx"] == test_user) &
        (train_data["rating"] >= 4)
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
    .head(20)
)

print("\nVALIDATION positives:")
print(
    val_data[
        (val_data["user_idx"] == test_user) &
        (val_data["rating"] >= 4)
        ][["recipe_idx", "rating"]]
    .sort_values("rating", ascending=False)
)

print("\nNCF recommendations:")
print(recommendations)

USER: 11035

TRAIN positives:
        recipe_idx  rating
47149        20558       5
68631        29598       5
140204       18618       5
150407       31352       5
180905        5148       5
208172       24785       5
225912       34917       5
223129        3051       5
247382       21395       5
251110       16502       5
349517        7898       5
260935       15623       5
281563       20762       5
325091       25905       5
374948       20470       5
355556        3799       5
207414       12145       4
378217       21850       4

VALIDATION positives:
       recipe_idx  rating
1           19836       5
1807         8096       5
4427        31147       5
12225       35167       5
30494       34303       4
32210       31172       4

NCF recommendations:
[ 457  237  593  410  143 2332  959  600 2181  216]


In [66]:
print("Users:", num_users)
print("Recipes:", num_recipes)
print("Train ratings:", len(train_data))
print("Validation ratings:", len(val_data))

print(
    "Average train ratings/user:",
    len(train_data) / num_users
)

print(
    "Average train ratings/recipe:",
    len(train_data) / num_recipes
)

Users: 17034
Recipes: 40027
Train ratings: 385260
Validation ratings: 42807
Average train ratings/user: 22.617118703768934
Average train ratings/recipe: 9.625003122892048


In [67]:
for user in eval_users[:5]:

    recs = fast_recommend_ncf(
        user_idx=user,
        model=model,
        num_recipes=num_recipes,
        user_seen_recipes=user_seen_recipes,
        top_k=10
    )

    print(f"User {user}: {recs}")

User 11035: [ 457  237  593  410  143 2332  959  600 2181  216]
User 1753: [2181  410  593 2332  600  237  420  143 1041  311]
User 2349: [ 237  457  593  959  410 2332  600  143 2181 1115]
User 586: [2181  410  117  593  420 2332  678  600 2912  143]
User 773: [ 2181   117   822 13158 13608   410  6815  1712  3627 19540]


In [68]:
from sklearn.decomposition import TruncatedSVD

# tfidf_matrix već imamo iz Content-Based modela

print("Original TF-IDF shape:", tfidf_matrix.shape)

svd = TruncatedSVD(
    n_components=64,
    random_state=42
)

recipe_content_features = svd.fit_transform(tfidf_matrix)

print("Reduced content shape:", recipe_content_features.shape)

NameError: name 'tfidf_matrix' is not defined

In [70]:
[name for name in dir() if not name.startswith("_")]

['BPRDataset',
 'DataLoader',
 'Dataset',
 'In',
 'NCF',
 'Out',
 'RecipeRatingDataset',
 'TruncatedSVD',
 'avg_loss',
 'avg_test_loss',
 'avg_train_loss',
 'avg_val_loss',
 'best_model_state',
 'best_val_loss',
 'bpr_dataset',
 'bpr_loader',
 'bpr_loss',
 'copy',
 'criterion',
 'device',
 'elapsed',
 'epoch',
 'epochs_without_improvement',
 'eval_users',
 'exit',
 'fast_recommend_ncf',
 'get_ipython',
 'hit_counts',
 'hits',
 'i',
 'loss',
 'model',
 'ncf_train',
 'negative_recipes',
 'negative_scores',
 'nn',
 'np',
 'num_epochs',
 'num_recipes',
 'num_users',
 'open',
 'optimizer',
 'patience',
 'pd',
 'positive_recipes',
 'positive_scores',
 'precision',
 'precision_at_10',
 'precisions',
 'predictions',
 'pydev_jupyter_vars',
 'quit',
 'rank',
 'ratings',
 'ratings_batch',
 'recall',
 'recall_at_10',
 'recalls',
 'recipe',
 'recipe_embeddings',
 'recipe_ids',
 'recipe_idx',
 'recipe_ranks',
 'recipe_tensor',
 'recipe_to_idx',
 'recipe_to_index',
 'recipes_batch',
 'recommend_ncf',

In [71]:
print("recipe type:", type(recipe))

if hasattr(recipe, "shape"):
    print("recipe shape:", recipe.shape)

if hasattr(recipe, "columns"):
    print("recipe columns:", recipe.columns.tolist())

recipe type: <class 'int'>


In [72]:
print("ratings type:", type(ratings))

if hasattr(ratings, "shape"):
    print("ratings shape:", ratings.shape)

if hasattr(ratings, "columns"):
    print("ratings columns:", ratings.columns.tolist())

ratings type: <class 'torch.Tensor'>
ratings shape: torch.Size([256])


In [73]:
from pathlib import Path

project_path = Path(r"C:\Users\Nikola\Desktop\FoodRecommendationSystem")

for file in project_path.rglob("*"):
    if file.is_file() and file.suffix.lower() in [".csv", ".json", ".parquet"]:
        print(file)

C:\Users\Nikola\Desktop\FoodRecommendationSystem\datasets\RAW_interactions.csv
C:\Users\Nikola\Desktop\FoodRecommendationSystem\datasets\RAW_recipes.csv
C:\Users\Nikola\Desktop\FoodRecommendationSystem\ai\datasets\processed\filtered_ratings.csv
C:\Users\Nikola\Desktop\FoodRecommendationSystem\ai\datasets\processed\test_ratings.csv
C:\Users\Nikola\Desktop\FoodRecommendationSystem\ai\datasets\processed\train_ratings.csv
C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\etc\jupyter\jupyter_notebook_config.d\jupyterlab.json
C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\etc\jupyter\jupyter_server_config.d\jupyter-lsp-jupyter-server.json
C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\etc\jupyter\jupyter_server_config.d\jupyterlab.json
C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\etc\jupyter\jupyter_server_config.d\jupyter_server_terminals.json
C:\Users\Nikola\Desktop\FoodRecommendationSystem\venv\etc\jupyter\jupyter_server_config.d\notebook.json
C:\Users\Nikola\De

In [74]:
import torch
import torch.nn as nn
import numpy as np
import copy
import time


class NCFv2(nn.Module):

    def __init__(
            self,
            num_users,
            num_recipes,
            content_embeddings,
            embedding_dim=32
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.recipe_embedding = nn.Embedding(
            num_recipes,
            embedding_dim
        )

        # Content embedding koji smo već napravili
        self.register_buffer(
            "content_embeddings",
            torch.tensor(
                content_embeddings,
                dtype=torch.float32
            )
        )

        content_dim = content_embeddings.shape[1]

        # User + recipe + content
        input_dim = embedding_dim * 2 + content_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.recipe_embedding.weight, std=0.01)

    def forward(self, user_ids, recipe_ids):

        user_vec = self.user_embedding(user_ids)

        recipe_vec = self.recipe_embedding(recipe_ids)

        content_vec = self.content_embeddings[recipe_ids]

        x = torch.cat(
            [
                user_vec,
                recipe_vec,
                content_vec
            ],
            dim=1
        )

        return self.mlp(x).squeeze(1)

In [75]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model_v2 = NCFv2(
    num_users=num_users,
    num_recipes=num_recipes,
    content_embeddings=recipe_embeddings,
    embedding_dim=32
).to(device)

print(
    "V2 parameters:",
    sum(p.numel() for p in model_v2.parameters())
)

Using device: cpu
V2 parameters: 1848737


C:\Users\Nikola\AppData\Local\Temp\ipykernel_13760\3293310792.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(


In [76]:
class BPRDatasetV2(torch.utils.data.Dataset):

    def __init__(
            self,
            ratings,
            num_recipes
    ):
        self.num_recipes = num_recipes

        positive = ratings[
            ratings["rating"] >= 4
            ][["user_idx", "recipe_idx"]].drop_duplicates()

        self.users = positive["user_idx"].values
        self.positive_items = positive["recipe_idx"].values

        # Sve što je korisnik već ocenio
        self.user_seen = {}

        for user, recipe in zip(
                ratings["user_idx"],
                ratings["recipe_idx"]
        ):
            if user not in self.user_seen:
                self.user_seen[user] = set()

            self.user_seen[user].add(recipe)

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):

        user = int(self.users[idx])
        positive = int(self.positive_items[idx])

        # Random negative
        negative = np.random.randint(
            0,
            self.num_recipes
        )

        while negative in self.user_seen[user]:
            negative = np.random.randint(
                0,
                self.num_recipes
            )

        return (
            user,
            positive,
            negative
        )

In [77]:
bpr_dataset_v2 = BPRDatasetV2(
    train_ratings,
    num_recipes
)

bpr_loader_v2 = torch.utils.data.DataLoader(
    bpr_dataset_v2,
    batch_size=1024,
    shuffle=True,
    num_workers=0
)

print("Training samples:", len(bpr_dataset_v2))

Training samples: 408440


In [78]:
optimizer_v2 = torch.optim.Adam(
    model_v2.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

epochs = 35
patience = 7

best_loss = float("inf")
best_state = None
no_improvement = 0


for epoch in range(epochs):

    model_v2.train()

    start_time = time.time()

    total_loss = 0
    batches = 0

    for users, positives, negatives in bpr_loader_v2:

        users = users.to(device)
        positives = positives.to(device)
        negatives = negatives.to(device)

        optimizer_v2.zero_grad()

        positive_scores = model_v2(
            users,
            positives
        )

        negative_scores = model_v2(
            users,
            negatives
        )

        # BPR loss
        loss = -torch.mean(
            torch.log(
                torch.sigmoid(
                    positive_scores - negative_scores
                ) + 1e-8
            )
        )

        loss.backward()

        optimizer_v2.step()

        total_loss += loss.item()
        batches += 1

    avg_loss = total_loss / batches

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch+1:02d}/{epochs} "
        f"- BPR Loss: {avg_loss:.4f} "
        f"- Time: {elapsed:.1f}s"
    )

    # Early stopping na training loss
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_state = copy.deepcopy(
            model_v2.state_dict()
        )
        no_improvement = 0

        print("  -> New best model!")

    else:
        no_improvement += 1

        print(
            f"  -> No improvement "
            f"({no_improvement}/{patience})"
        )

        if no_improvement >= patience:
            print("Early stopping triggered.")
            break


# Vrati najbolji model
model_v2.load_state_dict(best_state)

print("\nBest BPR loss:", best_loss)

Epoch 01/35 - BPR Loss: 0.4948 - Time: 16.3s
  -> New best model!
Epoch 02/35 - BPR Loss: 0.4246 - Time: 15.0s
  -> New best model!
Epoch 03/35 - BPR Loss: 0.3947 - Time: 14.0s
  -> New best model!
Epoch 04/35 - BPR Loss: 0.3676 - Time: 15.0s
  -> New best model!
Epoch 05/35 - BPR Loss: 0.3429 - Time: 17.1s
  -> New best model!
Epoch 06/35 - BPR Loss: 0.3249 - Time: 21.2s
  -> New best model!
Epoch 07/35 - BPR Loss: 0.3070 - Time: 22.6s
  -> New best model!
Epoch 08/35 - BPR Loss: 0.2943 - Time: 19.7s
  -> New best model!
Epoch 09/35 - BPR Loss: 0.2825 - Time: 21.5s
  -> New best model!
Epoch 10/35 - BPR Loss: 0.2727 - Time: 24.6s
  -> New best model!
Epoch 11/35 - BPR Loss: 0.2634 - Time: 22.4s
  -> New best model!
Epoch 12/35 - BPR Loss: 0.2565 - Time: 21.9s
  -> New best model!
Epoch 13/35 - BPR Loss: 0.2494 - Time: 21.0s
  -> New best model!
Epoch 14/35 - BPR Loss: 0.2452 - Time: 20.7s
  -> New best model!
Epoch 15/35 - BPR Loss: 0.2382 - Time: 18.2s
  -> New best model!
Epoch 16/3

In [80]:
print(type(val_users))
print(type(val_data))

if hasattr(val_users, "shape"):
    print("val_users shape:", val_users.shape)

if hasattr(val_data, "shape"):
    print("val_data shape:", val_data.shape)

print("val_users first:", val_users[:5])

<class 'numpy.ndarray'>
<class 'pandas.DataFrame'>
val_users shape: (10711,)
val_data shape: (42807, 7)
val_users first: [ 1628 11035  1753  2349   586]


In [81]:
# ==========================================
# BPR V2 - EVALUATION
# ==========================================

model.eval()

precision_scores = []
recall_scores = []
hit_counts = []

for user in eval_users:

    # User -> internal index
    if user not in user_to_idx:
        continue

    user_idx = user_to_idx[user]

    # Recepti koje je korisnik već video u treningu
    seen = user_seen_recipes.get(user_idx, set())

    # Relevantni recepti iz validation skupa
    user_val = val_data[
        val_data["user_id"] == user
        ]

    # Samo pozitivне оцене (4 или 5)
    relevant = set(
        user_val[
            user_val["rating"] >= 4
            ]["recipe_idx"].tolist()
    )

    if len(relevant) == 0:
        continue

    # Kandidati: svi recepti koje korisnik nije video
    candidates = [
        r for r in range(num_recipes)
        if r not in seen
    ]

    user_tensor = torch.tensor(
        [user_idx] * len(candidates),
        dtype=torch.long,
        device=device
    )

    recipe_tensor = torch.tensor(
        candidates,
        dtype=torch.long,
        device=device
    )

    with torch.no_grad():
        scores = model(
            user_tensor,
            recipe_tensor
        ).cpu().numpy()

    # Top 10
    top_indices = np.argsort(scores)[-10:][::-1]

    recommended = [
        candidates[i]
        for i in top_indices
    ]

    # Hits
    hits = len(
        set(recommended) & relevant
    )

    precision_scores.append(
        hits / 10
    )

    recall_scores.append(
        hits / len(relevant)
    )

    hit_counts.append(hits)


print("\n==============================")
print("BPR V2 EVALUATION")
print("==============================")

print(
    f"Users evaluated: {len(precision_scores)}"
)

print(
    f"Precision@10: "
    f"{np.mean(precision_scores):.4f}"
)

print(
    f"Recall@10:    "
    f"{np.mean(recall_scores):.4f}"
)

print(
    f"Average Hits: "
    f"{np.mean(hit_counts):.4f}"
)


BPR V2 EVALUATION
Users evaluated: 58
Precision@10: 0.0017
Recall@10:    0.0014
Average Hits: 0.0172


In [82]:
# ==========================================
# BPR V2 - ISPRAVNA EVALUACIJA
# Isto pravilo za sve korisnike
# ==========================================

model_v2.eval()

precision_scores = []
recall_scores = []
hit_counts = []

for user in val_users:

    # originalni user ID -> internal index
    if user not in user_to_idx:
        continue

    user_idx = user_to_idx[user]

    # Validation pozitivni recepti
    user_val = val_data[
        (val_data["user_id"] == user) &
        (val_data["rating"] >= 4)
        ]

    relevant = set(
        user_val["recipe_idx"].astype(int).tolist()
    )

    if len(relevant) == 0:
        continue

    # Recepti koje je korisnik već imao u TRAIN skupu
    train_seen = train_ratings[
        train_ratings["user_id"] == user
        ]["recipe_idx"].astype(int).tolist()

    seen = set(train_seen)

    # Kandidati
    candidates = np.array([
        r for r in range(num_recipes)
        if r not in seen
    ], dtype=np.int64)

    # Score kandidata u batch-evima
    scores = []

    with torch.no_grad():

        for start in range(0, len(candidates), 4096):

            batch_recipes = candidates[
                            start:start + 4096
                            ]

            user_tensor = torch.full(
                (len(batch_recipes),),
                user_idx,
                dtype=torch.long,
                device=device
            )

            recipe_tensor = torch.tensor(
                batch_recipes,
                dtype=torch.long,
                device=device
            )

            batch_scores = model_v2(
                user_tensor,
                recipe_tensor
            ).cpu().numpy()

            scores.append(batch_scores)

    scores = np.concatenate(scores)

    # Top 10
    top10_idx = np.argpartition(
        scores,
        -10
    )[-10:]

    top10_idx = top10_idx[
        np.argsort(scores[top10_idx])[::-1]
    ]

    recommended = candidates[top10_idx]

    # Hits
    hits = len(
        set(recommended) & relevant
    )

    precision_scores.append(
        hits / 10.0
    )

    recall_scores.append(
        hits / len(relevant)
    )

    hit_counts.append(hits)


print("\n==============================")
print("BPR V2 EVALUATION")
print("==============================")

print(
    f"Users evaluated: "
    f"{len(precision_scores)}"
)

print(
    f"Precision@10: "
    f"{np.mean(precision_scores):.6f}"
)

print(
    f"Recall@10: "
    f"{np.mean(recall_scores):.6f}"
)

print(
    f"Average Hits: "
    f"{np.mean(hit_counts):.6f}"
)

print(
    f"Users with at least 1 hit: "
    f"{sum(h > 0 for h in hit_counts)}"
)


BPR V2 EVALUATION
Users evaluated: 77
Precision@10: 0.000000
Recall@10: 0.000000
Average Hits: 0.000000
Users with at least 1 hit: 0


In [83]:
print("num_users:", num_users)
print("num_recipes:", num_recipes)

print("\nMappings:")
print("user_to_idx:", len(user_to_idx))
print("recipe_to_idx:", len(recipe_to_idx))

print("\nTRAIN columns:")
print(train_ratings.columns.tolist())

print("\nVAL columns:")
print(val_data.columns.tolist())

print("\nExample train:")
print(train_ratings.head())

print("\nExample validation:")
print(val_data.head())

num_users: 17034
num_recipes: 40027

Mappings:
user_to_idx: 17034
recipe_to_idx: 40027

TRAIN columns:
['user_id', 'recipe_id', 'date', 'rating', 'review', 'user_idx', 'recipe_idx']

VAL columns:
['user_id', 'recipe_id', 'date', 'rating', 'review', 'user_idx', 'recipe_idx']

Example train:
   user_id  recipe_id        date  rating  \
0   450571      38067  2009-11-21       5   
1   785604      31925  2008-05-18       5   
2   573325     213987  2011-08-28       5   
3   245429     110408  2006-10-24       5   
4    56112      76930  2004-08-25       4   

                                              review  user_idx  recipe_idx  
0  This could almost be used as chip dip.  Really...         0           0  
1  YUM! I made this recipe tonight to have with a...         1           1  
2  BK, this recipe is simply awesome! The muffins...         2           2  
3  I used 2 boxes of frozen spinach which is 20 o...         3           3  
4  This was my first attempt at this type dish, a... 

In [84]:
print("\nUSER 11035 CHECK")

print("Original user ID:", 11035)
print("Mapped user:", user_to_idx.get(11035))

print("\nValidation recipes:")
print(
    val_data[
        (val_data["user_id"] == 11035) &
        (val_data["rating"] >= 4)
        ][["user_id", "recipe_idx", "rating"]]
)


USER 11035 CHECK
Original user ID: 11035
Mapped user: None

Validation recipes:
Empty DataFrame
Columns: [user_id, recipe_idx, rating]
Index: []


In [85]:
user = 11035
recipe = 8096

user_idx = user_to_idx[user]

model_v2.eval()

with torch.no_grad():

    score = model_v2(
        torch.tensor(
            [user_idx],
            dtype=torch.long,
            device=device
        ),
        torch.tensor(
            [recipe],
            dtype=torch.long,
            device=device
        )
    ).item()

print("User:", user)
print("User idx:", user_idx)
print("Recipe:", recipe)
print("Model score:", score)

KeyError: 11035